🧸
# ¡¡¡ ExoSLAM 2025 !!!  
# NIRISS/SOSS Data Analysis

This notebook will walk you through the basics of data analysis for NIRISS/SOSS. This is a streamlined version of the exoTEDRF SOSS tutorial that can be found on the readthedocs webpage [here](https://exotedrf.readthedocs.io/en/latest/content/notebooks/tutorial_niriss-soss.html). There are also tutorials for NIRSpec and MIRI timeseries available there too.   
A broad overview of the exoTEDRF (pronounced exo-tedorf) functionalities are summarized in its JOSS paper, [Radica (2024)](https://ui.adsabs.harvard.edu/abs/2024JOSS....9.6898R/abstract). 

In [ ]:
# First thing to do is set your CRDS variables so we can download the most up-to-date JWST 
# reference files. 

import os
os.environ['CRDS_PATH'] = './crds_cache'
os.environ['CRDS_SERVER_URL'] = 'https://jwst-crds.stsci.edu'

Now let's get some data! We're going to use the WASP-39b transit observations from the JWST Transiting Exoplanet Community Early Release Science program published in [Feinstein & Radica et al. (2023)](https://ui.adsabs.harvard.edu/abs/2023Natur.614..670F/abstract). You can find and download the relevant data products from [Box](https://uchicago.box.com/s/ezx57mwux6r1584efbv3i53fc9aot1oq) (~7Gb total).  
Once the Box data products are downloaded, unzip the exoSLAM 2025 folder (if not done automatically), and put it into the same directory as this notebook.

<div class="alert alert-block alert-info"> <b>NOTE:</b> Throughout this tutorial we will be producing and saving outputs to disk. In order to save all outputs, you'll need ~30Gb of free storage. If you don't have that amount of storage available, you can always remove the output files from earlier steps once they have been run (e.g., remove the early Stage 1 outputs before running Stage 2).

Now let's create output directories to store our data products.

In [ ]:
from exotedrf import utils

# Generate output directories

utils.verify_path('DMS_uncal')
utils.verify_path('pipeline_outputs_directory')
utils.verify_path('pipeline_outputs_directory/Stage1')
utils.verify_path('pipeline_outputs_directory/Stage2')
utils.verify_path('pipeline_outputs_directory/Stage3')
utils.verify_path('pipeline_outputs_directory/Stage4')

And finally, let's unpack the files we downloaded from Box into the locations where exoTEDRF is going to expect them to be.   
This involves putting the raw "uncal" file into the DMS_uncal directory, and the "badpixstep" data files into the Stage2 directory. The reasons for this will become clear later, so just trust me for now.  
We'll also pull the two other files out of the downloaded directory, and get rid of it -- I don't like having unneccesary directories cluttering my workspace. 

In [ ]:
import shutil
import os

# Move uncal file to DMS_uncal.

shutil.move('exoSLAM 2025/jw01366001001_04101_00001-seg004_nis_uncal.fits', 'DMS_uncal/')

# Move badpixstep files to pipeline_outputs_directory/Stage2.

shutil.move('exoSLAM 2025/jw01366001001_04101_00001-seg001_nis_badpixstep.fits', 'pipeline_outputs_directory/Stage2/')
shutil.move('exoSLAM 2025/jw01366001001_04101_00001-seg002_nis_badpixstep.fits', 'pipeline_outputs_directory/Stage2/')
shutil.move('exoSLAM 2025/jw01366001001_04101_00001-seg003_nis_badpixstep.fits', 'pipeline_outputs_directory/Stage2/')

# Move other calibration files to current directory.

shutil.move('exoSLAM 2025/model_background256.npy', './')
shutil.move('exoSLAM 2025/F277W.npy', './')

# Remove unneeded directory.

os.rmdir('exoSLAM 2025')

Before diving in, here's a brief look at the workflow we're going to follow. exoTEDRF is subdivided into four stages, broadly mirroring those of the STScI jwst pipeline.  
The stages are as follows:
 - Stage 1: Detector Level Processing (i.e., ramps-to-slopes)
 - Stage 2: Spectroscopic Processing (i.e., slope calibrations)
 - Stage 3: 1D Spectral Extraction (i.e., images-to-spectra)
 - Stage 4: Light Curve Fitting (not covered here)
 
This notebook will walk you through stages 1 to 3, or from raw data to extracted stellar spectra, mostly following the steps laid out in [Radica et al. (2023)](https://ui.adsabs.harvard.edu/abs/2023MNRAS.524..835R/abstract), with updates from [Fournier-Tondreau et al. (2024)](https://ui.adsabs.harvard.edu/abs/2024MNRAS.528.3354F/abstract) and [Radica et al. (2024)](https://ui.adsabs.harvard.edu/abs/2024ApJ...962L..20R/abstract). So please read those articles for a more in-depth discussion of how some of the following steps work "under the hood". 

Most of these steps are identical if you're also analyzing NIRSpec observations. Most of the peculiarities of dealing with NIRISS vs NIRSpec data is handled under the hood. But, in several places I'll point out things that would need to be tweaked for NIRSpec data. You can also check out the exoTEDRF NIRSpec tutorial on readthedocs [here](https://exotedrf.readthedocs.io/en/latest/content/notebooks/tutorial_nirspec-g395h.html) to explicitly see the NIRSpec workflow. The main peculiarity with NIRSpec is that certain gratings (e.g., G395H) are split across two separate detectors, which have their own associated data files. So each step would have to be run twice, once for the NRS1 detector files and once for NRS2. However, the lower-resolution options (e.g., PRISM, G395M) only use the NRS1 detector, and so don't require this.  
MIRI is a bit more different, but the Eureka! tutorial is focusing on MIRI, so I won't say anything more about it. 

## Stage 1 -- Detector-Level Processing

Stage 1 performs multiple important calibrations on the raw, 4D (integrations, groups, y-pixel, x-pixel) data cubes. Several steps are simply wrappers around the existing functionalities of the STScI pipeline, whose documentation can be found [here](https://jwst-pipeline.readthedocs.io/en/latest/jwst/pipeline/calwebb_detector1.html).

In [ ]:
from exotedrf import stage1

# First define some input and output directory paths.

uncal_indir = 'DMS_uncal/'  # Where our uncalibrated files are found. 
outdir_s1 = 'pipeline_outputs_directory/Stage1/'  # Where to save our Stage 1 output files.

Now let's make a list with all the uncal input files.  
In this case, the full TSO is broken up into four segments as you can see in the plot below.  
![bears](files/Quicklook_Lightcurve.pdf)  
This is a quick look at the raw light curve (you can already clearly see the decrease in flux as the planet transits the host star). The grey dashed lines delineate the separations between segments. Segments are simply just a means to keep the size of the data manageable and don't really have any scientific meaning.  
Since there are four segments in this observation, there should be four corresponding uncal files, however, you only downloaded one. That's because for this tutorial, we're only going to process the final segment, segment 4. The other files that you put into the Stage 2 directory, the "badpixstep" ones, are the already-calibrated versions of the first three segments, so that at the end of this, we'll still be able to extract spectra from the entire observation.  
Nothing really changes in our workflow because we're only using a single segment of the observation -- things will just run faster. If there are any differences that would need to be accounted for when running the full four segments, I'll point them out. In any case, you can also refer to the readthedocs tutorial linked above to see how things would look (spoiler alert, it's basically the same). 

In [ ]:
# Our list of input files just has the final (fourth) segment in the observation. 

filenames = [uncal_indir+'jw01366001001_04101_00001-seg004_nis_uncal.fits']

Let's check the readout pattern of the observations. 

In [ ]:
from astropy.io import fits

with fits.open(filenames[0]) as file:
    print('No. integrations: {}'.format(file[0].header['NINTS']))
    print('No. groups: {}'.format(file[0].header['NGROUPS']))
    print('subarray: {}'.format(file[0].header['SUBARRAY']))

Here we can see that the full TSO consists of 537 integrations, with 9 groups per integration using the 256x2048 pixel subarray, which provides the full 0.6 -- 2.8µm wavelength range. 

Let's now take a look at what the raw data looks like.

In [ ]:
import matplotlib.pyplot as plt

with fits.open(filenames[0]) as file:
    plt.figure(figsize=(8, 3))
    # Let's check out the last group of the 10th integration
    plt.imshow(file[1].data[10, -1], aspect='auto', origin='lower',  
               vmin=1e4, vmax=3e4)
    plt.xlabel('Disperson Axis', fontsize=12)
    plt.ylabel('Cross-dispersion Axis', fontsize=12)
    plt.show()

We can clearly see the first (lower) and second (upper) orders of the target spectrum, as well as a number of nasty detector effects! 

We're now ready to start with the actual reduction.

### DQ Initalization Step
Intializes the data quality flags (i.e., flag know bad, hot, warm, etc. pixels).   

In [ ]:
# Initialize the step by passing the input files and the directory to which we want to save 
# the outputs. 
# There's also an option to pass a custom hot pixel map in order to identify and flag hot pixels 
# which are not already in the default data quality map. This should be a boolean map with True 
# values for hot pixels and False otherwise. A version of this is output by the BadPixStep towards 
# the end of Stage 2. 

step = stage1.DQInitStep(filenames, hot_pixel_map=None, output_dir=outdir_s1)

# Now run the step! 
# Assuming you've never run any version of a jwst pipeline before, this step is going to download a whole load of 
# files from the crds. These are all necessary context and reference files for the data, so don't worry, this is 
# supposed to happen. 

results = step.run(save_results=True, force_redo=True)

### Saturation Detection Step
Flags any saturated pixels in the observations.  

In [ ]:
# This time we pass the outputs from the previous step as the inputs for this step.

step = stage1.SaturationStep(results, output_dir=outdir_s1)

results = step.run(save_results=True, force_redo=True)

### Superbias Subtraction Step
Now we subtract the superbias level from all frames.   
The detector bias (or "superbias" in jwst parlance -- I'm not entirely sure why) is the level of base read-out noise in each pixel. It also isn't constant across the entire detector, but varies from pixel to pixel.  

<div class="alert alert-block alert-info"> <b>NOTE:</b> When running the Superbias Correction with the direct outputs of a previous step (i.e., with save_results=False) a specific error may occur: <br>
   <code>UFuncTypeError: Cannot cast ufunc 'subtract' output from dtype('float32') to dtype('uint16') with casting rule 'same_kind'. </code> <br>
This likely stems from a crds reference file compatability issue, and is being looked into. In the mean time, a workaround is to simply create a list of paths to the previous step output files, e.g., <br>
    <code> results = ['XXXX_seg001_saturationstep.fits', 'XXXX_seg002_saturationstep.fits', ...]</code> <br>  
and pass this to the SuperBiasStep.</div>

In [ ]:
step = stage1.SuperBiasStep(results, output_dir=outdir_s1)

# This step has options for diagnostic plotting. Specifying do_plot=True will create the step 
# plot and save it to the output directory.

results = step.run(save_results=True, force_redo=True, do_plot=True)

In [ ]:
# Display the step plot.

from IPython.display import Image

Image(filename=outdir_s1 + 'superbiasstep_1.png') 

Nine random frames are shown in the diagnostic plot, where we can verify that many of the detector effects we saw above, including the bright clouds at the bottom of the frame are now gone.

### 1/f Noise Correction Step
1/f noise is time-correlated noise which manifests in SOSS observations as column-correlated "stripes". The 1/f noise is actually caused by time-varying voltage levels as the detector is read, and as such it is actually one of the very last noise sources (along with read noise) to be injected into the data. exoTEDRF, therefore, corrects 1/f noise at the group-level (i.e., before ramp fitting) as is done with the other JWST instrument modes.  

1/f noise correction can also be done at the integration-level (i.e., after ramp fitting) in addition to, or instead of the group-level correction is your wish. In which case, feel free to skip this step!

#### First Background Subtraction
For reasons outlined more completely in [Feinstein & Radica et al. (2023)](https://ui.adsabs.harvard.edu/abs/2023Natur.614..670F/abstract) and [Radica et al. (2023)](https://ui.adsabs.harvard.edu/abs/2023MNRAS.524..835R/abstract), the SOSS background signal **must** be subtracted before attempting the 1/f noise correction. Here, we do a group-level background subtraction --- that is we subtract a background signal separately from each group. 

Technically the background signal needs to be corrected for flat field, and linearity effects (amongst other things) which we have not yet done. We'll therefore be adding the background signal back to the data after the 1/f noise correction.

The SOSS background has a unique structure caused by the undispersed light from the zodiacal background falling off the JWST pick-off mirror. STScI has created a model of the SOSS background which we already downloaded! 

If we were analyzing NIRSpec data here, we would skip the background step and proceed straight to the 1/*f* correction.

In [ ]:
# Open the STScI background model.

import numpy as np

background_model = np.load('model_background256.npy')

Let's see what the background model looks like.

In [ ]:
plt.figure(figsize=(8, 3))
plt.imshow(background_model, aspect='auto', origin='lower')
plt.show()

There's clearly a sharp increase in the background level at pixel ~700 caused by the zodi order 0 suddenly falling off the pick-off mirror. The background level slowly then decreases towards the right.

Let's now subtract the SOSS background.

In [ ]:
# The background correction step technically lives within Stage 2 of exoTEDRF.

from exotedrf import stage2

# In addition to the input files and output directory, we also want to pass the background model
# that we opened above, as well as an estimate of the integrations which define the transit 
# baseline. From our quick look light curve above, the first 150 and last 100 are reasonable estimates for 
# the ingress and egress integrations. However, since we're only dealing with the last segment, we'll just 
# use the last 20 integrations as the baseline.

step = stage2.BackgroundStep(results, background_model=background_model, 
                             baseline_ints=[-20], output_dir=outdir_s1)

# Specifying differential=True allows for the background model to be scaled separately to the 
# right and left of the "step" at pixel ~700 (recommended). Note, however, that for SUBSTRIP96 
# observations (the bright star subarray) this functionality is not supported.

results, bkg = step.run(save_results=True, differential=True, force_redo=True, do_plot=True)

This step will produce two diagnostic plots. The first, once again, shows nine random frames, but now with the background subtracted. The increase in brightness at column ~700 that we saw after the superbias subtraction has nicely been removed. In some cases (like here), we can even see evidence for the 1/f striping!

In [ ]:
Image(filename=outdir_s1 + 'backgroundstep_1.png') 

The second plot shows a cut along a detector row at the top of the detector to better visualize the background removal. 

In [ ]:
Image(filename=outdir_s1 + 'backgroundstep_2.png') 

#### 1/*f* Noise Correction
We can now proceed with the correction of the 1/*f* noise itself. 

exoTEDRF provides four different methods for 1/*f* correction, but we'll proceed with the simplest one, "scale-achromatic", here. It simply consists of constructing difference images by subtracting a median stack from each individual frame, and subtracting the median value from each column. 

It's a good idea to disclude any bad pixels, or pixels which are affected by field stars from the calculation so that we don't bias the 1/*f* value calculated for a given column. To address this, there is an optional ```pixel_masks``` keyword to which can be passed masks of pixels to exclude in each frame.   
If no such masks are available (like for this tutorial), the step will automatically mask all pixels with non-zero DQ flags, as well as obvious outliers.  

exoTEDRF produces pixel masks for this specific purpose during the RampFitStep in Stage 1 and TracingStep in Stage 2, so it is always a good idea to do a first quick run through of the pipeline to get a feel for the data, and create these useful auxiliary files. Then do a second "good" reduction using the outputs from the first pass.

In [ ]:
pixel_masks = None

# If available, the exoTEDRF pixel masks will have the filenames XXX_pixelflags.fits and there
# will always be one for each segment.

For the scale-achromatic method, it is also a good idea to pass an estimate of the planet's white light curve. This is so that the median stack can be scaled to the average flux level of each frame before being subtracted.  
Again, if no such estimate is available, the light curve will be estimated from the current data. A light curve estimate .npy file is also created during the TracingStep.

In [ ]:
timeseries = None

# If available, the exoTEDRF timeseries estimate file will have the name XXX_lcestimate.npy.

In [ ]:
# Initialize the step passing the input files as well as any auxiliary files mentioned above.
# Moreover, we also want to indicate the indices of the baseline integrations and the 1/f 
# correction method that we wish to use -- we'll use the same values as above.
# Lastly, we also pass the background model from the BackgroundStep so that it can be re-added. 

step = stage1.OneOverFStep(results, baseline_ints=[-20], output_dir=outdir_s1, 
                           pixel_masks=pixel_masks, soss_timeseries=timeseries,
                           method='scale-achromatic', soss_background=bkg)

# even_odd_rows=True calculates seperate 1/f noise values for even- and odd-values rows. 
# inner_mask_width=40 defines the width of pixels around the target spectral trace to mask. It's
# a good idea to make this at least as large as your planned extraction aperture. 

results = step.run(save_results=True, even_odd_rows=True, force_redo=True, do_plot=True, 
                   soss_inner_mask_width=40)

At least two diagnostic plots are produced here (depending on the method). The first will always be our familiar nine-panel plot, now showing difference images after the 1/*f* noise correction. We don't want to see any column-correlated noise left here!

In [ ]:
Image(filename=outdir_s1 + 'oneoverfstep_1.png') 

The second is a power series of the data before and after the 1/*f* correction. The power spectral density before the correction clearly increases, roughly linearly, towards shorter frequencies (hence 1/*f*!). After the correction though, we can see that the psd is much flatter. We have done a pretty good job of removing all the 1/*f* noise!

In [ ]:
Image(filename=outdir_s1 + 'oneoverfstep_2.png') 

<div class="alert alert-block alert-info"> <b>NOTE:</b> If running the OneOverFStep at the integration level, don't pass anything to the background keyword so that the background is not unnecessarily re-added to the data. </div>

### Linearity Step
The HgCdTe detectors used in NIRISS are not perfectly linear, which means that as saturation is approached, the counts begin to plateau and less flux is recorded than is actually observed. This is not an issue just with NIRISS, but all other JWST instruments as well. This step applies pre-calcuated polynomials to correct these non-linearity effects in the ramp. 

In [ ]:
step = stage1.LinearityStep(results, output_dir=outdir_s1)

results = step.run(save_results=True, force_redo=True, do_plot=True)

The first diagnostic plot shows (mean-subtracted) group-to-group differences. For a perfect non-linearity correction, we would expect each difference to be identical. The correction is therefore not perfect, but it's much better than where we started!

In [ ]:
Image(filename=outdir_s1 + 'linearitystep_1.png') 

The second plot shows the residuals to a perfectly straight ramp before and after correction. before the correction, we see large positive residuals at intermediate group numbers, indicating that the ramp is indeed plateauing. We're much better, but again not perfect, after the correction.

In [ ]:
Image(filename=outdir_s1 + 'linearitystep_2.png') 

### Jump Detection Step
We now want to detect and flag cosmic ray hits in the data. There are two main ways to do this: up-the-ramp or time-domain flagging. 

The up-the-ramp flagging algorithm is the default algorithm in the STScI pipeline. In entails identifying discontinuities above a certain threshold in ramps, and flagging these as jumps. Unfortunately, this method can be quite temperamental and has been found by many studies to flag random noise. It also cannot be applied to ngroup=2 datasets (which are reasonably common with NIRISS/SOSS!)

The time-domain flagging method uses a sigma clipping algorithm to identify cosmic ray hits. This method also has the benefit of working for observations with any number of groups. 

In [ ]:
step = stage1.JumpStep(results, output_dir=outdir_s1)

# Here we will use the time-domain rejection by specifying flag_in_time=True, and use a clipping 
# threshold of 7 sigma.
# In order to use the up-the-ramp flagging, specify instead flag_up_ramp=True and pass an 
# appropriate value to the rejection_threshold keyword. 

results = step.run(save_results=True, force_redo=True, flag_in_time=True, 
                   time_rejection_threshold=7, do_plot=True)

The diagnostic plot here shows the locations of flagged jumps (i.e., cosmic ray hits), as well as hot and other bad pixels in nine frames. 

In [ ]:
Image(filename=outdir_s1 + 'jump.png') 

### Ramp Fit Step
Now that all the initial detector-level calibrations are done, we are ready for ramp fitting! This step is a wrapper around the STScI pipeline step that fits a slope and intercept as a function of group to each pixel. We don't use the intercept values, and only care about the slopes moving forwards.

This step also produces outlier pixel masks by combining all existing data flags. One flag file is produced for each segment and is saved in the Stage 1 directory with the name XXX_pixelflags.fits.

In [ ]:
step = stage1.RampFitStep(results, output_dir=outdir_s1)

results = step.run(save_results=True, force_redo=True)

We are now ready to start with Stage 2!

## Stage 2 -- Spectroscopic Processing

This stage performs some additional calibrations on the, now 3D (integrations, y-pixel, x-pixel), data after the ramp fitting to make the data ready for the spectral extraction. Once again, information on the default STScI pipeline steps can be found [here](https://jwst-pipeline.readthedocs.io/en/latest/jwst/pipeline/calwebb_spec2.html).

In [ ]:
from exotedrf import stage2

# Let's now save outputs to the Stage 2 subdirectory. 

outdir_s2 = 'pipeline_outputs_directory/Stage2/'

### Assign WCS Step
This is a wrapper around the STScI pipeline step which assigns the appropriate WCS to the data. 

In [ ]:
step = stage2.AssignWCSStep(results, output_dir=outdir_s2)

results = step.run(save_results=True, force_redo=True)

### Source Type Determination Step
For multiple subsequent steps to function, we need to correctly identify the source type in our observations. For all exoplanet TSOs, the correct type is a point source.

In [ ]:
step = stage2.SourceTypeStep(results, output_dir=outdir_s2)

results = step.run(save_results=True, force_redo=True)

### Flat Field Subtraction Step
This step corrects flat fielding effects (i.e., the fact that the detector pixels do not respond identically to illumination). 

In [ ]:
step = stage2.FlatFieldStep(results, output_dir=outdir_s2)

results = step.run(save_results=True, force_redo=True)

### Background Subtraction Step
It's now time to subtract the background once and for all! (Or for the first time if you skipped the group-level 1/*f* noise correction -- which you shouldn't have done in this tutorial). Now that the background flux has been flat fielded and correted for non-linearty effects, it can be safely removed. 

We're going to run the same step as we did previously in Stage 1, but this time on the slope images. This means we're only getting one background image as opposed to one for each group. 

For NIRSpec data, we'd skip this step, but perhaps repeat the OneOverFStep instead at the integration level. 

In [ ]:
# Reuse the same background model from the group-level correction

step = stage2.BackgroundStep(results, background_model=background_model, 
                             baseline_ints=[-20], output_dir=outdir_s2)

# Using all the same parameters from the group-level background subtraction again here, 
# including the differential scaling. 
# We don't need to re-add the background to the data later, so we only need to return the 
# corrected data.

results = step.run(save_results=True, differential=True, force_redo=True, do_plot=True)[0]

In [ ]:
Image(filename=outdir_s2 + 'backgroundstep_1.png') 

In [ ]:
Image(filename=outdir_s2 + 'backgroundstep_2.png') 

<div class="alert alert-block alert-info"> <b>NOTE:</b> If doing the OneOverFStep at the integration level, this is where the step should be run. </div>

### Bad Pixel Correction Step
We are now going to interpolate any remaining bad pixels in the data to be ready for the spectral extraction.  

The BadPixStep performs two iterations of bad pixel detection and correction: the first is a spatial correction, and the second a temporal one. 

The spatial correction uses a median stack of all integrations to identify any pixels which are systematic outliers across the entire time series. These pixels are flagged, and then interpolate using a median of the surrounding pixels in each integration. Any pixels with DQ flags are also interpolated in this way. It will also produce and save a map of these hot pixels which are not already in the default DQ flags. 

The temporal flagging works the same way as the time-domain jump detection, by flagging any pixels which are outliers along the time axis, and replacing them with a median of the surrounding pixels in time. 

In [ ]:
step = stage2.BadPixStep(results, baseline_ints=[-20], output_dir=outdir_s2)

# We're going to use a 10 sigma threshold for both the spatial and temporal corrections.

results = step.run(save_results=True, force_redo=True, do_plot=True, space_thresh=10, time_thresh=10, 
                   window_size=9)

In [ ]:
Image(filename=outdir_s2 + 'badpixstep.png') 

The diagnostic plot here shows the deep stack (median stack) with the locations of all hot and other flagged pixels. 

We've now caught up to the point to which the files from the first three segments had been calibrated. There are a couple more things that we need to do before extracting the spectra, but it's important that these final steps be run on the full observation together and not just a single segment. So for that reason, we're going to join up with the other three segments now.

In [ ]:
# Add the other three segments to the results list.

results = ['pipeline_outputs_directory/Stage2/jw01366001001_04101_00001-seg001_nis_badpixstep.fits',
           'pipeline_outputs_directory/Stage2/jw01366001001_04101_00001-seg002_nis_badpixstep.fits',
           'pipeline_outputs_directory/Stage2/jw01366001001_04101_00001-seg003_nis_badpixstep.fits',
           'pipeline_outputs_directory/Stage2/jw01366001001_04101_00001-seg004_nis_badpixstep.fits']

### PCA Reconstruction Step

This is the penultimate step of Stage 2, and the final one that actually performs any true data calibrations before the extraction. 

The PCAReconstructStep uses principle component analysis (PCA) to deconstruct the timeseries into eigenimages that explain the largest amount of variance in the data, and their corresponding eigenvector timeseries. We then have the option to reconstruct the data, removing the components which correspond to detector "noise" (e.g., trace position drifts, tilt events, etc.) -- which we aren't going to do here though. 

In [ ]:
# Now since we're using the full time series, we'll use the first 150 and last 100 integrations
# as the locations of ingress and egress.

step = stage2.PCAReconstructStep(results, baseline_ints=[150, -100], output_dir=outdir_s2)

# We're going to show the first 10 PCA components, but not remove any in this tutorial.

results, deepframe = step.run(save_results=True, force_redo=True, do_plot=True, pca_components=10, 
                              remove_components=None)

The diagnostic plot is the intial results of the PCA. Here we see the eigenimages (right) and eigenvalue timeseries (left) of the 10 components that explain the most variance in the data. 

The first component is the transit white light curve itself. The next three components are due to detector-based trends. The second and third being caused by a slight sub-pixel drift in the position of the trace and the fourth being a beating pattern caused by JWST's thermal control system. All higher order components are noise.

So in this case, we might want to remove components 2 -- 4 in the reconstruction, since these represent detector noise and not the astrophysical signal that we care about. But again, we're not going to do that in this tutorial. 

In [ ]:
Image(filename=outdir_s2 + 'stability_pca.png') 

You'll note though that the fifth eigenimage seems to show some weird structures as well. Namely, the so-called zeroth-order contaminants are picked out in the eigenimage, and the eigenvalue timeseries seems to kind of trace the shape of the transit. This is due to the fact that although the spectral trace itself (and its expended wings which cover the majority of the SOSS frame) "see" the planet's transit, the contaminants do not since they are just background stars.  

A good plan of attack is to run this step first, passing ```remove_components=None``` to see the initial results. In this case, the step will simply return the input data with no reconstruction. You can then re run the step specifying the components that you identified as detector-related noise and want to remove. 

<div class="alert alert-block alert-info"> <b>WARNING:</b> Its important to be careful with the number of components being removed. Just like with high resolution cross correlation spectroscopy, removing components *can* have an effect on the final atmosphere spectrum. Only remove things that you *know* are detector correlated (positional drifts, beating patterns, etc.). If in doubt, compare your level of light curve scatter as well as the end atmosphere spectrum itself with and without the PCA removal. 
</div>

### Tracing Step
This step does not do any actual data calibration, but produces a number of auxiliarly files which may be useful either for improving the reduction, or in the subsequent light curve analysis. 

It has three major functionalities:  
1. Locate the centroids of all three SOSS orders via the [edgetrigger algorithm](https://ui.adsabs.harvard.edu/abs/2022PASP..134j4502R/abstract).  
2. (optional -- SOSS only) Generate a mask of order 0 contaminants from background stars.  
3. (optional -- SOSS only) Create a smoothed estimate of the order 1 white light curve.  

The positions of the target traces are necessary for the spectral extraction, and this is the only functionality that happens by default.

The first optional functionality is to add the positions of order 0 contaminants to the pixel masks created above. Since SOSS is slitless, background stars can contaminate the frame, and potentially bias the 1/f noise estimation. Using an F277W exposure, the TracingStep automatically detects the locations of these contaminants and adds them to the pixel masks. We're not going to do this right now, because it isn't necessary for our purposes, but we'll still look at what this functionality does.

In [ ]:
# File names of pixel masks produced by the RampFitStep.
# We only ran the final segment through this step, so we actually don't have pixel mask files for 
# each segment, and trying to only pass one will cause the step to crash.

pixel_masks = None

We also need an F277W exposure for this to work. This is why we downloaded the F277W exposure data!

Let's open the F277W exposure and see what it looks like.

In [ ]:
# Open the F277W exposure.

f277w = np.load('F277W.npy')

# Now let's display it!

plt.figure(figsize=(8, 3))
plt.imshow(f277w, aspect='auto', origin='lower', vmin=-1, vmax=1)
plt.show()

Only the reddest end of the target order 1 spectral trace is transmitted by the F277W filter, and so all of the background star contaminants are revealed! Every splotch above is the undispersed 0th order of a background star which was within the field of view of the target during the observation. This is why it's a good idea to judiciously choose your appropriate APAs when planning NIRISS/SOSS observations!

It's even possible to have dispersed contaminats present on the detector, and there is actually a very faint one in this dataset! But these most likely won't show up in the F277W exposure.  

In any case, when running the order 0 flagging in the TracingStep, the locations of order 0 contaminants are identified in the F277W exposure and added to the pixel flags. 

The final optional functionality is to create an estimate of the order 1 white light curve. Of course, you can simply do this yourself after the spectral extraction!

NIRSpec observations don't make use of either of these two optional functionalities since there are no background contaminants in NIRSpec and we don't need a white light curve scaling for the 1/*f* correction.

In [ ]:
# Specify generate_order0_mask=True and pass the f277w exposure as well as the pixel masks to 
# add the positions of contaminats to the pixel masks. 
# generate_lc=True will produce the order 1 white light curve estimate.

step = stage2.TracingStep(results, deepframe=deepframe, generate_order0_mask=False, f277w=None, 
                          generate_lc=True, baseline_ints=[150, -100], output_dir=outdir_s2) 

centroids = step.run(pixel_flags=pixel_masks, save_results=True, force_redo=True, do_plot=True)

In [ ]:
Image(filename=outdir_s2 + 'centroiding.png') 

The diagnostic plot here shows the results of the trace centroiding for all three orders.

Now one more thing before moving on to the extraction. Let's take a look at the deep stack and check what we want the size of our extraction box to be.

In [ ]:
# Read in the trace positions.

import pandas as pd

centroids = pd.read_csv(outdir_s2 + 'jw01366001001_04101_00001_nis_centroids.csv', comment='#')

In [ ]:
# Read in the deepframe.

deepframe = fits.getdata(deepframe)

In [ ]:
# Display the deepframe and the trace positions.

plt.figure(figsize=(8, 3))
plt.imshow(deepframe, aspect='auto', origin='lower', vmin=0, vmax=10)

# Show a width of 30 pixels around the trace.

plt.plot(centroids['xpos'], centroids['ypos o1']+15, ls='--', c='blue', label='Order 1')
plt.plot(centroids['xpos'], centroids['ypos o1']-15, ls='--', c='blue')
plt.plot(centroids['xpos'], centroids['ypos o2']+15, ls='--', c='red', label='Order 2')
plt.plot(centroids['xpos'], centroids['ypos o2']-15, ls='--', c='red')

plt.legend()
plt.ylim(0, 255)
plt.show()

Everything looks good! We're now ready for Stage 3 and the spectral extraction.

## Stage 3 -- 1D Spectral Extraction

This is the shortest stage, as it just performs the 1D spectral extraction. 

In [ ]:
from exotedrf import stage3

# Let's now save outputs to the Stage 3 subdirectory. 

outdir_s3 = 'pipeline_outputs_directory/Stage3/'

There are two methods that one can use for the spectral extraction: a simple box aperture extraction or the ATOCA algorithm. The box aperture extraction is the easiest, simply place an aperture around the target trace and sum up the flux! 

Now, you probably noticed that the order 1 and order 2 traces partially overlap in the deep stakc image we made above. So a simple box aperture extraction will have some degree of self-contamination! It turns out for *relative measurements*, like transit observations, this effect is neglible for the vast majority of targets, so a box extraction is probably a safe bet. However, the ATOCA algorithm was developed to explicitly model and correct the SOSS self-contamination. If you're interested, the algorithm is detailed in [Darveau-Bernier et al. (2022)](https://ui.adsabs.harvard.edu/abs/2022PASP..134i4502D/abstract) and [Radica et al. (2022)](https://ui.adsabs.harvard.edu/abs/2022PASP..134j4502R/abstract).

In this case, we're simply going to go with the box aperture extraction for convenience. 

In [ ]:
# We're passing the parameters of the WASP-39 host star in order to cross correlate the 
# extracted stellar spectrum with a PHOENIX model and improve the wavelength solution.

step = stage3.Extract1DStep(results, extract_method='box', output_dir=outdir_s3, 
                            st_teff=5485, st_logg=4.453, st_met=0.01)

# Here, we're using an aperture width of 30 pixels. 
# We also pass the trace positions from above.

results = step.run(extract_width=30, centroids=centroids, save_results=True, force_redo=True)

*Tada!* You now have stellar spectra of WASP-39! Let's take a quick look at the wavelength-dependent light curves.

## Inspect Outputs

In [ ]:
# Open the extracted spectrum file and get the relevant quantities.

with fits.open(outdir_s3 + 'WASP-39_box_spectra_fullres.fits') as spec:
    
    wave1 = spec[1].data[5:-5]  # Order 1 wavelengths
    wave2 = spec[5].data  # Order 2 wavelengths
    
    order1 = spec[3].data[:, 5:-5]  # Order 1 spectra
    order2 = spec[7].data  # Order 2 spectra
    
base = np.concatenate([np.arange(150), np.arange(100)-100]).astype(int) # Baseline integrations
    
# Normalize the extracted spectra.  

order1_norm = order1 / np.nanmedian(order1[base], axis=0)
order2_norm = order2 / np.nanmedian(order2[base], axis=0)

In [ ]:
from exotedrf.plotting import make_2d_lightcurve_plot

# For order 2, only the wavelengths from 0.6 -- 0.85µm are useable.

ii = np.where((wave2 >= 0.6) & (wave2 < 0.85))[0]

# Display the light curves.

kwargs = {'vmin': 0.9725, 'vmax': 1.01}
make_2d_lightcurve_plot(wave1, order1_norm, wave2[ii], order2_norm[:, ii], **kwargs)

Look at how clean those are!  

If you look closely, you can notice the blues are darker, that is the transits are deeper, around 1.2, 1.4, and 1.8µm. Those are prominant water absorption bands; you can literally see the water absorption in the light curves! This is transit spectroscopy, *by eye!!* How cool is that?

exoTEDRF Stages 1 to 3 can also be run in script form via the provided run_DMS.py file. Simply fill out the corresponding yaml file with all relevant inputs, and you're good to go! Generally, I like to take a first pass at the data in a notebook, where I can double check the outputs of each step, and perhaps dig a bit deeper into some interesting things that pop up. I'll then use the script for the second pass.

You're now ready to fit some light curves and get your atmosphere spectrum, which is what you really want in the end, isn't it?! But we're not going to do that here unfortunately, so that'll be left as an exercise for the reader, as they say. 